# Build Kvasir-VQA x1 metadata + image manifest

Prep pipeline for the Kvasir-VQA x1 dataset: download images, write manifests/metadata, and record basic stats.

In [1]:
import os
from pathlib import Path
import random

import pandas as pd
from PIL import Image as PILImage
from tqdm.auto import tqdm

from datasets import load_dataset, get_dataset_infos, Image as HFImage


In [2]:
# Config
HF_DATASET = "SimulaMet/Kvasir-VQA-x1"  
SPLITS = ["train", "validation", "test"]  # if dataset has single split, will fall back to that
MAX_SAMPLES_PER_SPLIT = int(os.getenv("MAX_SAMPLES_PER_SPLIT", "0")) or None  # set to int/env for smoke test

SPLIT_SEED = 42
TRAIN_FRAC = 0.8
VAL_FRAC = 0.1
SPLIT_BY_IMAGE = True  # avoid leakage across questions for same image

OUT_ROOT = Path("./out")
IMAGES_DIR = OUT_ROOT / "images"
META_DIR = OUT_ROOT / "metadata"
MANIFEST_DIR = OUT_ROOT / "manifests"
for d in [IMAGES_DIR, META_DIR, MANIFEST_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RAW_META_CSV = META_DIR / "metadata_raw.csv"
ENRICHED_META_CSV = META_DIR / "metadata_enriched.csv"
IMAGE_MANIFEST_CSV = MANIFEST_DIR / "image_manifest.csv"

print("Dataset:", HF_DATASET)
print("Output root:", OUT_ROOT)


Dataset: SimulaMet/Kvasir-VQA-x1
Output root: out


In [3]:
def safe_img_id(ex, split: str, idx: int) -> str:
    # Try common keys for image filename
    for k in ["image_id", "img_id", "id", "filename"]:
        if k in ex and ex[k]:
            return str(ex[k]).split("/")[-1].split(".")[0]
    return f"{split}_{idx:06d}"


def save_image(img, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    if hasattr(img, "mode") and img.mode != "RGB":
        img = img.convert("RGB")
    img.save(path, format="JPEG")


def ensure_image_column(ds):
    # Cast string/bytes image column to decoded PIL images when needed
    if "image" in ds.column_names and not isinstance(ds.features["image"], HFImage):
        return ds.cast_column("image", HFImage())
    return ds


In [4]:
# Determine splits by inspecting the dataset dict
try:
    ds_all = load_dataset(HF_DATASET)
    ds_all = {k: ensure_image_column(v) for k, v in ds_all.items()}
    use_splits = list(ds_all.keys())
    print("Found splits:", use_splits)
except Exception as e:
    print("Failed to load dataset without split:", e)
    ds_all = None
    use_splits = ["train"]


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/14.9M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.65M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/143594 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/15955 [00:00<?, ? examples/s]

Found splits: ['train', 'test']


In [5]:
manifest_rows = []
meta_rows = []

for split in use_splits:
    print(f"Processing split: {split}")
    try:
        if ds_all is not None:
            ds = ds_all[split]
        else:
            ds = load_dataset(HF_DATASET, split=split)
        ds = ensure_image_column(ds)
    except Exception as e:
        print(f"Split {split} unavailable: {e}")
        continue
    print("  Split size:", len(ds))
    if len(ds) == 0:
        continue
    if MAX_SAMPLES_PER_SPLIT is not None:
        max_n = min(MAX_SAMPLES_PER_SPLIT, len(ds))
        ds = ds.select(range(max_n))
        print(f"  Truncated to {max_n} samples")

    for idx, ex in enumerate(ds):
        img = ex.get("image")
        if isinstance(img, str):
            # Fallback for raw URLs/paths if column wasn't decoded
            try:
                img = PILImage.open(img).convert("RGB")
            except Exception:
                pass
        if img is None:
            img_path_field = None
            for k in ["image_path", "img_path", "path", "file_name", "file_path"]:
                if k in ex and ex[k]:
                    img_path_field = ex[k]
                    break
            if img_path_field is None:
                continue
            try:
                img = PILImage.open(img_path_field).convert("RGB")
            except Exception:
                continue
        if img is None:
            continue

        img_id = safe_img_id(ex, split, idx)
        out_path = IMAGES_DIR / split / f"{img_id}.jpg"
        if not out_path.exists():
            save_image(img, out_path)

        question = ex.get("question")
        answer = ex.get("answer")
        question_type = ex.get("question_type")
        answer_type = ex.get("answer_type")
        question_class = ex.get("question_class")
        complexity = ex.get("complexity")
        original = ex.get("original")

        manifest_rows.append({
            "split": split,
            "img_id": img_id,
            "image_path": str(out_path),
            "exists": out_path.exists(),
        })
        meta_rows.append({
            "split": split,
            "img_id": img_id,
            "image_path": str(out_path),
            "question": question,
            "answer": answer,
            "question_type": question_type,
            "answer_type": answer_type,
            "question_class": question_class,
            "complexity": complexity,
            "original": original,
            "orig_height": getattr(img, "height", None),
            "orig_width": getattr(img, "width", None),
        })

manifest = pd.DataFrame(manifest_rows)
meta = pd.DataFrame(meta_rows)


Processing split: train
  Split size: 143594


AttributeError: 'str' object has no attribute 'mode'

In [ ]:
# Preserve raw metadata before split assignment
meta_raw = meta.copy()

# Create train/val/test splits if dataset doesn't already include them
if 'split' not in meta.columns or meta['split'].nunique() <= 1:
    if not (0 < TRAIN_FRAC < 1 and 0 < VAL_FRAC < 1 and TRAIN_FRAC + VAL_FRAC < 1):
        raise ValueError('Check TRAIN_FRAC/VAL_FRAC: must sum to < 1')
    rng = random.Random(SPLIT_SEED)
    img_ids = sorted(meta['img_id'].dropna().unique())
    rng.shuffle(img_ids)
    n = len(img_ids)
    n_train = int(n * TRAIN_FRAC)
    n_val = int(n * VAL_FRAC)
    train_ids = set(img_ids[:n_train])
    val_ids = set(img_ids[n_train:n_train + n_val])

    def assign_split(img_id):
        if img_id in train_ids:
            return 'train'
        if img_id in val_ids:
            return 'validation'
        return 'test'

    if SPLIT_BY_IMAGE:
        meta['split'] = meta['img_id'].apply(assign_split)
    else:
        idx = list(range(len(meta)))
        rng.shuffle(idx)
        n_train_rows = int(len(idx) * TRAIN_FRAC)
        n_val_rows = int(len(idx) * VAL_FRAC)
        labels = (['train'] * n_train_rows +
                  ['validation'] * n_val_rows +
                  ['test'] * (len(idx) - n_train_rows - n_val_rows))
        meta['split'] = ''
        meta.loc[idx, 'split'] = labels

    # Update manifest to match new splits (image_path stays the same)
    split_map = meta.drop_duplicates('img_id').set_index('img_id')['split'].to_dict()
    manifest['split'] = manifest['img_id'].map(split_map)
    print('Assigned splits by img_id:', meta['split'].value_counts().to_dict())
else:
    print('Using existing splits:', meta['split'].value_counts().to_dict())


Assigned splits by img_id: {'train': 46966, 'test': 5952, 'validation': 5931}


In [ ]:
manifest.to_csv(IMAGE_MANIFEST_CSV, index=False)
meta_raw.to_csv(RAW_META_CSV, index=False)
meta.to_csv(ENRICHED_META_CSV, index=False)

print("Saved manifest:", IMAGE_MANIFEST_CSV, "rows:", len(manifest))
print("Saved metadata:", RAW_META_CSV, "rows:", len(meta))
if len(meta) > 0:
    print("Records per split:")
    print(meta.groupby("split").size())
else:
    print("No records written; check dataset/source settings.")


Saved manifest: out/manifests/image_manifest.csv rows: 58849
Saved metadata: out/metadata/metadata_raw.csv rows: 58849
Records per split:
split
test           5952
train         46966
validation     5931
dtype: int64


In [ ]:
print("Nulls per column:")
print(meta.isnull().sum())


Nulls per column:
split                0
img_id               0
image_path           0
question             0
answer               0
question_type    58849
answer_type      58849
orig_height          0
orig_width           0
dtype: int64
